In [1]:
from budget import load_budget, extract_cashflow, aggregate_cashflow
from cpiu import load_cpiu

from plotly.offline import init_notebook_mode
init_notebook_mode(connected=True)
from plotly.subplots import make_subplots
from scipy.stats import norm
import plotly.express as px
import plotly.graph_objects as go
import polars as pl
import polars.selectors as cs

In [2]:
cpiu = load_cpiu()
budget = load_budget()

In [3]:
import datetime

if datetime.date.today() >= datetime.date(2026, 2, 1):
    raise Exception("Elevator expenditure priced in")
ELEVATOR_EXPENDITURE = 31247.13
ELEVATOR_EXPENDITURE_EXPR: pl.Expr = (
    pl.when(pl.col("month").is_between(1, 3))
    .then(ELEVATOR_EXPENDITURE / 3)
    .otherwise(0)
)

In [4]:
expenses = extract_cashflow(budget["expenses"])


# Simulate next year.
MONTHS_TO_SIMULATE = 12
SIMULATIONS = 10000
UNPAID_BILLS = 3146.42
# Based on the "Operation" account pulled from Daisy dashboard on 2026-01-10
STARTING_BALANCE = 23080.09 - UNPAID_BILLS
cashflow_by_month = (
    aggregate_cashflow(extract_cashflow(budget["incomes"]))
    .with_columns(
        pl.int_ranges(-3, 9)
        .alias("budget_increase")
        .list.eval((pl.element() / 100).round(2))
    )
    .explode("budget_increase")
    .with_columns(pl.col("amount") * (1 + pl.col("budget_increase")))
    .join(
        aggregate_cashflow(expenses)
        .join(cpiu, left_on=["year", "month"], right_on=["Year", "month"])
        .select("year", "month", pl.col("amount") * (1 + pl.col("eoy_delta"))),
        on=["year", "month"],
        how="full",
        suffix="_expense",
    )
    .select(
        "budget_increase",
        "year",
        "month",
        pl.col("amount").fill_null(0) - pl.col("amount_expense").fill_null(0),
    )
)
monte_carlo_simulations = (
    cashflow_by_month.group_by("budget_increase")
    .agg(pl.col("amount"))
    .with_columns(simulation_id=pl.int_ranges(0, SIMULATIONS))
    .explode("simulation_id")
    .with_columns(pl.col("amount").list.sample(MONTHS_TO_SIMULATE, shuffle=True))
    .explode("amount")
    .with_columns(
        (pl.row_index("month").over("budget_increase", "simulation_id") + 1).cast(
            pl.UInt8
        ),
        pl.col("amount").cum_sum().over("budget_increase", "simulation_id")
        + STARTING_BALANCE,
    )
    .with_columns(pl.col("amount") - ELEVATOR_EXPENDITURE_EXPR)
)

In [5]:
monthly_cashflow_by_budget_increase = (
    cashflow_by_month.sort("month")
    .pivot("month", index=["budget_increase", "year"], values="amount")
    .drop("year")
)
from functools import reduce


covariant_year_simulations = (
    reduce(
        lambda df, c: df.join(
            monthly_cashflow_by_budget_increase.select("budget_increase", c),
            on="budget_increase",
        ),
        monthly_cashflow_by_budget_increase.select(cs.digit()).columns,
        monthly_cashflow_by_budget_increase.select("budget_increase"),
    )
    .with_columns(pl.row_index("simulation_id").over("budget_increase"))
    .unpivot(
        cs.digit(),
        index=["budget_increase", "simulation_id"],
        variable_name="month",
        value_name="amount",
    )
    .sort("budget_increase", "simulation_id", pl.col("month").str.to_integer())
    .with_columns(
        pl.col("month").str.to_integer(dtype=pl.UInt8),
        pl.col("amount").cum_sum().over("budget_increase", "simulation_id")
        + STARTING_BALANCE,
    )
    .with_columns(pl.col("amount") - ELEVATOR_EXPENDITURE_EXPR)
)

In [6]:
def line_charts(simulations: pl.DataFrame) -> go.Figure:
    simulations_by_month = (
        simulations.group_by("month", "budget_increase")
        .agg(
            amount_average=pl.col("amount").mean(),
            amount_min=pl.col("amount").min(),
            amount_max=pl.col("amount").max(),
        )
        .sort("budget_increase", "month")
    )

    TRENDLINE_FIG_COL_COUNT = 2

    unique_budget_increases = simulations["budget_increase"].unique()

    fig = make_subplots(
        rows=int(len(unique_budget_increases) / TRENDLINE_FIG_COL_COUNT),
        cols=TRENDLINE_FIG_COL_COUNT,
        shared_yaxes="all",
        subplot_titles=[
            f"budget_increase={budget_increase}"
            for budget_increase in unique_budget_increases
        ],
    )
    for i, budget_increase in (
        unique_budget_increases.to_frame().with_row_index().iter_rows()
    ):
        sims = simulations_by_month.filter(pl.col("budget_increase") == budget_increase)
        row = int(i / TRENDLINE_FIG_COL_COUNT) + 1
        col = i % TRENDLINE_FIG_COL_COUNT + 1
        fig.add_trace(
            go.Scatter(
                x=pl.concat([sims["month"], sims["month"].reverse()]),
                y=pl.concat([sims["amount_min"], sims["amount_max"].reverse()]),
                name=f"min/max {budget_increase}",
                fill="toself",
            ),
            row=row,
            col=col,
        )
        fig.add_trace(
            go.Scatter(x=sims["month"], y=sims["amount_average"], name=budget_increase),
            row=row,
            col=col,
        )
        fig.update_xaxes(title_text="month", row=row, col=col)
    fig.update_layout(height=1000, legend=go.layout.Legend(title="budget_increase"))
    return fig


line_charts(monte_carlo_simulations).update_layout(
    title_text="Monte Carlo Simulations"
).show()

In [7]:
line_charts(covariant_year_simulations).update_layout(
    title_text="Covariant Year Simulations"
).show()

In [8]:
def pie_charts(simulations: pl.DataFrame) -> go.Figure:
    simulation_mins = simulations.group_by("budget_increase", "simulation_id").agg(
        pl.col("amount").min()
    )
    fig = px.pie(
        simulation_mins.group_by(
            "budget_increase",
            ruinous=pl.col("amount") < 0,
        )
        .len("simulation_count")
        .with_columns(
            ruinous=pl.when("ruinous")
            .then(pl.lit("Special Assessment"))
            .otherwise(pl.lit("Safe"))
        )
        .sort("budget_increase"),
        names="ruinous",
        values="simulation_count",
        facet_col="budget_increase",
        facet_col_wrap=3,
        color_discrete_sequence=["#4B08AF", "#32965D"],
        height=1000,
    )
    return fig


pie_charts(monte_carlo_simulations).update_layout(
    title="Likelihood of Special Assessment (Monte Carlo)",
).show(renderer="notebook_connected")

In [9]:
pie_charts(covariant_year_simulations).update_layout(
    title="Likelihood of Special Assessment (Covariant Year Simulations)",
).show(renderer="notebook_connected")

# Special Assessment
95% confidence that the special assessment--if there is one--will be less than `amount`

In [10]:
# Inflation estimation from https://www.federalreserve.gov/monetarypolicy/files/fomcprojtabl20250917.pdf
INFLATION = 0.026
simulation_mins_by_type = (
    pl.concat(
        [
            monte_carlo_simulations.select(
                "budget_increase", "simulation_id", "month", "amount"
            ).with_columns(simulation_type=pl.lit("Monte Carlo")),
            covariant_year_simulations.with_columns(
                pl.col("simulation_id").cast(pl.Int64),
                simulation_type=pl.lit("Covariant Year"),
            ),
        ]
    )
    .with_columns(
        pl.col("simulation_type").cast(pl.Enum(["Monte Carlo", "Covariant Year"]))
    )
    .group_by("budget_increase", "simulation_id", "simulation_type")
    .agg(pl.col("amount").min())
)
assessment_recommendations = (
    simulation_mins_by_type.filter(pl.col("amount") < 0)
    .sort("budget_increase")
    .with_columns(-pl.col("amount"))
    .group_by("budget_increase", "simulation_type")
    .agg(
        amount_95_conf=pl.col("amount").mean()
        + pl.col("amount").std() * norm.ppf(0.95),
        amount_99_conf=pl.col("amount").mean()
        + pl.col("amount").std() * norm.ppf(0.99),
    )
    .with_columns(
        amount_95_conf_with_inflation=pl.col("amount_95_conf") * (1 + INFLATION),
        amount_99_conf_with_inflation=pl.col("amount_99_conf") * (1 + INFLATION),
    ).sort('budget_increase','simulation_type')
)
assessment_recommendations.style

budget_increase,simulation_type,amount_95_conf,amount_99_conf,amount_95_conf_with_inflation,amount_99_conf_with_inflation
-0.03,Monte Carlo,22037.615180397646,27051.708262470936,22610.593175087986,27755.05267729518
-0.03,Covariant Year,15823.490698994743,19723.765982991754,16234.901457168608,20236.58389854954
-0.02,Monte Carlo,20638.977034467156,25399.716004761063,21175.590437363302,26060.10862088485
-0.02,Covariant Year,13889.038570249017,17402.198686395328,14250.153573075491,17854.655852241605
-0.01,Monte Carlo,19447.356261168443,23918.481517010092,19952.987523958822,24540.362036452356
-0.01,Covariant Year,12002.389996943779,15133.501562450772,12314.452136864318,15526.972603074493
0.0,Monte Carlo,17681.493164626212,21810.97494953635,18141.211986906495,22378.060298224296
0.0,Covariant Year,10166.98747254633,12921.154167598961,10431.329146832535,13257.104175956534
0.01,Monte Carlo,17006.7067857163,21004.9377746085,17448.881162144924,21551.066156748322
0.01,Covariant Year,9960.987603253157,12360.01921913108,10219.97328093774,12681.379718828488


In [11]:
fig = px.bar(
    expenses.with_columns(
        chart_date=pl.date(pl.col("date").dt.year(), pl.col("date").dt.month(), 1)
    ),
    x="chart_date",
    y="amount",
    color="headline_category",
)
fig.show(renderer="notebook_connected")

In [12]:
fig = px.bar(
    expenses.with_columns(
        chart_date=pl.date(pl.col("date").dt.year(), pl.col("date").dt.month(), 1)
    ),
    x="chart_date",
    y="amount",
    color="category",
)
fig.show(renderer="notebook_connected")